In [1]:
import tensorflow as tf
import os

# ---- FIXED PATH ----
DATA_DIR = r"sorted_images"  # <- make sure this folder exists in your working directory

IMG_SIZE = (28, 28)
BATCH_SIZE = 32

def main():
    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(f"Dataset folder not found: {DATA_DIR}")

    # 1. Load dataset from directory
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        color_mode="grayscale",
        batch_size=BATCH_SIZE
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        color_mode="grayscale",
        batch_size=BATCH_SIZE
    )

    class_names = train_ds.class_names
    print("Classes:", class_names)


if __name__ == "__main__":
    main()

Found 27201 files belonging to 36 classes.
Using 21761 files for training.
Found 27201 files belonging to 36 classes.
Using 5440 files for validation.
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [2]:
import tensorflow as tf

ds = tf.keras.utils.image_dataset_from_directory(
    r"sorted_images",
    image_size=(64, 64),
    batch_size=32,
    color_mode="grayscale"
)

print(ds.class_names)

Found 27201 files belonging to 36 classes.
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [3]:
import os
import json
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# === CONFIG ===
DATA_DIR = r"sorted_images"   # <-- single dataset
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
EPOCHS = 10
MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"


def load_dataset(path):
    """Load a dataset from a single directory."""
    return tf.keras.utils.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="training",
        seed=123,
        image_size=IMG_SIZE,
        color_mode="grayscale",
        batch_size=BATCH_SIZE
    ), tf.keras.utils.image_dataset_from_directory(
        path,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMG_SIZE,
        color_mode="grayscale",
        batch_size=BATCH_SIZE
    )


def main():
    if not os.path.isdir(DATA_DIR):
        raise FileNotFoundError(f"Directory not found: {DATA_DIR}")

    print(f"\nLoading dataset from: {DATA_DIR}")
    train_ds, val_ds = load_dataset(DATA_DIR)

    # Extract class names
    class_names = train_ds.class_names
    num_classes = len(class_names)

    print("\nCLASSES:", class_names)
    print("Total classes:", num_classes)

    # Save label map
    label_map = {i: name for i, name in enumerate(class_names)}
    with open(LABELS_PATH, "w") as f:
        json.dump(label_map, f)

    # Performance optimizations
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(2000).prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)

    # Build model
    model = keras.Sequential([
        layers.Rescaling(1./255, input_shape=IMG_SIZE + (1,)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.summary()

    # Train
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS
    )

    # Save model
    model.save(MODEL_PATH)
    print(f"\nModel saved to {MODEL_PATH}")
    print(f"Label map saved to {LABELS_PATH}")


if __name__ == "__main__":
    main()


Loading dataset from: sorted_images
Found 27201 files belonging to 36 classes.
Using 21761 files for training.
Found 27201 files belonging to 36 classes.
Using 5440 files for validation.

CLASSES: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Total classes: 36


C:\Users\sinch\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\preprocessing\tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)                │ (None, 64, 64, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │       1,179,904 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 36)                  │           9,252 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 1,281,828 (4.89 MB)

 Trainable params: 1,281,828 (4.89 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 80s 59ms/step - accuracy: 0.3693 - loss: 2.0620 - val_accuracy: 0.8257 - val_loss: 0.6148
Epoch 2/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.7959 - loss: 0.6828 - val_accuracy: 0.9061 - val_loss: 0.3248
Epoch 3/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.8692 - loss: 0.4317 - val_accuracy: 0.9351 - val_loss: 0.2293
Epoch 4/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.8976 - loss: 0.3165 - val_accuracy: 0.9443 - val_loss: 0.1941
Epoch 5/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.9191 - loss: 0.2636 - val_accuracy: 0.9386 - val_loss: 0.2004
Epoch 6/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 50ms/step - accuracy: 0.9280 - loss: 0.2249 - val_accuracy: 0.9513 - val_loss: 0.1593
Epoch 7/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 50ms/step - accuracy: 0.9386 - loss: 0.1846 - val_accuracy: 0.9544 - val_loss: 0.1539
Epoch 8/10
681/681 ━━━━━━━━━━━━━━━━━━━━ 34s 49ms/step - accuracy: 0.9483 - loss: 0.1550 - 


Model saved to char_digit_model.h5
Label map saved to label_map.json


In [5]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\2\img003-002.png"

    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...
Processing image: C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\2\img003-002.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step
Predicted: 2
Speaking: The predicted number is two.


In [6]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\6\img007-020.png"

    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...


Processing image: C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\6\img007-020.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
Predicted: 6
Speaking: The predicted number is six.


In [7]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\b\img012-012.png"

    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...


Processing image: C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\b\img012-012.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 264ms/step
Predicted: b
Speaking: The predicted character is b.


In [8]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\h\img018-020.png"

    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...


Processing image: C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\h\img018-020.png


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 342ms/step
Predicted: h
Speaking: The predicted character is h.


In [13]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\s\img055-040.png"
    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...


Processing image: C:\Users\sinch\OneDrive\Desktop\dcproject\sorted_images\s\img055-040.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step
Predicted: s
Speaking: The predicted character is s.


In [1]:
import json
import sys
import numpy as np
from PIL import Image
import tensorflow as tf
import pyttsx3

MODEL_PATH = "char_digit_model.h5"
LABELS_PATH = "label_map.json"
IMG_SIZE = (64, 64)

# Spoken digit names for 0–9
DIGIT_WORDS = {
    "0": "zero",
    "1": "one",
    "2": "two",
    "3": "three",
    "4": "four",
    "5": "five",
    "6": "six",
    "7": "seven",
    "8": "eight",
    "9": "nine",
}

def load_model_and_labels():
    """Load the trained model and label map."""
    model = tf.keras.models.load_model(MODEL_PATH)

    with open(LABELS_PATH, "r") as f:
        label_map = json.load(f)

    label_map = {int(k): v for k, v in label_map.items()}
    return model, label_map


def preprocess_image(image_path):
    """Convert image to grayscale, resize, and shape for model."""
    img = Image.open(image_path).convert("L")
    img = img.resize(IMG_SIZE)

    img_array = np.array(img, dtype="float32")
    img_array = np.expand_dims(img_array, axis=-1)
    img_array = np.expand_dims(img_array, axis=0)
    return img_array


def speak_text(text):
    engine = pyttsx3.init()
    engine.say(text)
    engine.runAndWait()


def main():
    # YOUR IMAGE PATH
    image_path = r"C:\Users\sinch\OneDrive\Desktop\testdata.jpeg"
    print("Loading model...")
    model, label_map = load_model_and_labels()

    print(f"Processing image: {image_path}")
    img_array = preprocess_image(image_path)

    preds = model.predict(img_array)
    pred_idx = int(np.argmax(preds[0]))
    pred_char = label_map[pred_idx]

    # Choose speech based on digit or letter
    if pred_char in DIGIT_WORDS:
        speech = f"The predicted number is {DIGIT_WORDS[pred_char]}."
    else:
        speech = f"The predicted character is {pred_char}."

    print(f"Predicted: {pred_char}")
    print(f"Speaking: {speech}")

    speak_text(speech)


if __name__ == "__main__":
    main()

Loading model...


Processing image: C:\Users\sinch\OneDrive\Desktop\testdata.jpeg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 567ms/step
Predicted: 0
Speaking: The predicted number is zero.
